In [1]:
import pandas as pd

# ==============================
# 1. Cargar CSV desde GitHub
# ==============================

url = "https://raw.githubusercontent.com/iJahir/etl-data-pipeline-25168282020-/refs/heads/main/data/raw/D_empleados.csv"

df = pd.read_csv(url)

print("Primeros registros:")
print(df.head())


# ==============================
# 2. Limpieza de datos
# ==============================

empleados = df.copy()

# limpiar nombres de columnas
empleados.columns = empleados.columns.str.strip().str.lower()

# limpiar espacios
for col in empleados.select_dtypes(include='object').columns:
    empleados[col] = empleados[col].astype(str).str.strip()

# eliminar duplicados
empleados = empleados.drop_duplicates()


# ==============================
# 3. Separar válidos y rechazados
# ==============================

validos = empleados[
    (~empleados.astype(str).apply(lambda x: x.str.contains("error|text")).any(axis=1)) &
    (empleados.notna().all(axis=1))
].copy()

rechazados = empleados.drop(validos.index).copy()


# ==============================
# 4. Motivo de rechazo
# ==============================

def motivo(row):
    motivos = []

    for col in empleados.columns:
        if pd.isna(row[col]):
            motivos.append(f"{col}_vacio")

        if "error" in str(row[col]):
            motivos.append(f"{col}_error")

        if "text" in str(row[col]):
            motivos.append(f"{col}_texto")

    return ",".join(motivos)


rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)


# ==============================
# 5. Exportar
# ==============================

validos.to_csv("curated_empleados.csv", index=False)
rechazados.to_csv("rejects_empleados.csv", index=False)

print("Empleados procesado correctamente ✅")

Primeros registros:
        id   nombre     dept
0      NaN  text_54    error
1  text_55       82      NaN
2      NaN      NaN      NaN
3      NaN    error  text_36
4      NaN      NaN    error
Empleados procesado correctamente ✅


In [3]:
from google.colab import files

files.download('curated_empleados.csv')
files.download('rejects_empleados.csv')
#

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>